# LiDAR 4방향 거리 (앞·뒤·좌·우)

강의 예제 "LiDAR 활용하기 - 4방향 거리 가져오기"를 바탕으로 수정했습니다.

## 하는 일
- 각 방향(0°, 90°, 180°, 270°) ±10° 범위의 거리 중앙값(mm)을 0.3초마다 갱신해 표시합니다.
- LiDAR 읽기는 별도 스레드, 화면 갱신은 메인 스레드에서 합니다. 중단은 ■(Interrupt) 버튼으로 합니다.

## 원본에서 바꾼 점
- 스캔 시작이 간헐적으로 실패해서 재시도를 추가했습니다 (정면 거리 노트북과 동일).
- 읽기 스레드가 오류로 멈추면 화면에 오류를 표시합니다. 원본은 조용히 멈춰서 `---`만 계속 나올 수 있었습니다.
- 종료할 때 읽기 스레드를 먼저 멈춘 뒤 LiDAR를 정리합니다.
- 값이 없는 방향은 이전 값을 유지하지 않고 `---`로 표시합니다.

## 확인한 것
- 손으로 한 방향씩 가리면 해당 라벨의 값만 줄어들어, 0°=정면, 90°=오른쪽, 180°=뒤, 270°=왼쪽 가정이 맞는 것을 확인했습니다.
- 거리 값 자체의 정확도는 정면 거리 노트북에서 자로 잰 값과 비교해 확인했습니다(거의 일치).

In [1]:
import time
import threading
import statistics
from rplidar import RPLidar, RPLidarException
from IPython.display import clear_output

PORT = '/dev/ttyUSB0'
BAUDRATE = 460800
ANGLE_WINDOW = 10        # 각 방향 중심 ± 도
UPDATE_INTERVAL = 0.3    # 거리 갱신 주기(초)
DIRECTIONS = {"Front": 0, "Right": 90, "Back": 180, "Left": 270}

dist = {name: None for name in DIRECTIONS}
status = {"msg": "시작 중..."}
error = None
stop_event = threading.Event()


def measures_with_retry(lidar, retries=5):
    """iter_measures()를 그대로 쓰되, 시작 단계에서 실패하면 정리 후 재시도."""
    for attempt in range(1, retries + 1):
        gen = lidar.iter_measures()
        try:
            first = next(gen)
        except RPLidarException as e:
            status["msg"] = f"스캔 시작 실패 ({attempt}/{retries}): {e}"
            try:
                lidar.stop()
            except Exception:
                pass
            time.sleep(1.0)
            continue
        status["msg"] = f"스캔 중 (시작 시도 {attempt}회)"
        yield first
        yield from gen
        return
    raise RuntimeError("LiDAR 스캔 시작에 계속 실패했습니다")


def in_angle_range(angle, center, window=ANGLE_WINDOW):
    diff = abs((angle - center + 180) % 360 - 180)
    return diff <= window


def lidar_worker(lidar):
    global error
    bufs = {name: [] for name in DIRECTIONS}
    last_update = time.time()
    try:
        for _, quality, angle, distance in measures_with_retry(lidar):
            if stop_event.is_set():
                break
            if distance <= 0:
                continue
            for name, center in DIRECTIONS.items():
                if in_angle_range(angle, center):
                    bufs[name].append(distance)
                    break
            now = time.time()
            if now - last_update > UPDATE_INTERVAL:
                for name in DIRECTIONS:
                    dist[name] = statistics.median(bufs[name]) if bufs[name] else None
                    bufs[name].clear()
                last_update = now
    except Exception as e:
        error = e


lidar = RPLidar(PORT, baudrate=BAUDRATE, timeout=2)
worker = threading.Thread(target=lidar_worker, args=(lidar,), daemon=True)
worker.start()

try:
    while worker.is_alive():
        time.sleep(UPDATE_INTERVAL)
        clear_output(wait=True)
        print("---- Distance Info ----")
        for name in DIRECTIONS:
            d = dist[name]
            print(f"{name:<5} : {d:.1f} mm" if d is not None else f"{name:<5} : ---")
        print("-----------------------")
        print(status["msg"])
    if error:
        print("LiDAR 스레드 오류:", error)

except KeyboardInterrupt:
    print("\nStopping lidar...")

finally:
    stop_event.set()
    worker.join(timeout=3)          # 읽기 스레드가 멈춘 뒤에 정리
    for fn in (lidar.stop, lidar.stop_motor, lidar.disconnect):
        try:
            fn()
        except Exception:
            pass
    print("Lidar stopped and disconnected.")

---- Distance Info ----
Front : 395.5 mm
Right : 5833.0 mm
Back  : 1423.5 mm
Left  : 3825.0 mm
-----------------------
스캔 중 (시작 시도 1회)

Stopping lidar...
Lidar stopped and disconnected.
